In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

In [2]:
from datasets import load_dataset
ds = load_dataset("llamafactory/alpaca_zh", cache_dir="../data",split = "train")

In [3]:
ds

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 51155
})

In [4]:
ds = ds.train_test_split(test_size=0.8,seed = 42)
ds

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 10231
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 40924
    })
})

In [5]:
ds['train'][5]

{'instruction': '解释气候变化对环境的两个影响',
 'input': '',
 'output': '气候变化对环境有几个影响。其中一个影响是全球平均气温上升，导致冰川融化、极端天气事件和海平面上升。另一个影响是由极端天气条件和升温引起的自然栖息地破坏导致的生物多样性减少。'}

## 2、数据预处理

In [6]:
#模型下载
# from modelscope import snapshot_download
# model_dir = snapshot_download('qwen/Qwen2-0.5B', cache_dir="./models")

In [7]:
model_path = "./models/qwen/Qwen2-0___5B"

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer

Qwen2TokenizerFast(name_or_path='./models/qwen/Qwen2-0___5B', vocab_size=151643, model_max_length=32768, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [9]:
def process_func(datasets, tokenizer, max_length=256):
    """
    处理输入示例，合并用户的指令和输入字段，然后对输入和输出进行分词。
    该函数准备好训练自回归语言模型的数据格式。

    参数：
    - datasets (dict): 包含 'instruction'、'input' 和 'output' 字段的字典。
    - tokenizer (AutoTokenizer): 用于分词的 tokenizer。
    - max_length (int): 序列的最大长度。

    返回：
    - dict: 准备好的训练数据，包括 tokenized 的输入和标签。
    """
    # 1. 生成输入部分，包括 "Human: " 和 "Assistant: " 标签
    if datasets['input'].strip() == "":
        combined_input = "用户: " + datasets['instruction'] + "\n\n 机器人: "
    else:
        combined_input = "用户: " + datasets['instruction'] + "\n" + datasets['input'] + "\n\n 机器人: "

    # 2. 生成输出部分，并在最后加上终止标记
    output_text = datasets["output"] + tokenizer.eos_token  # 加上 eos_token 标记回复的结束

    # 3. 合并输入和输出作为模型的完整输入序列
    full_input = combined_input + output_text

    # 4. 对完整输入进行分词处理，确保生成 input_ids 和 attention_mask
    encodings = tokenizer(
        full_input, 
        max_length=max_length, 
        truncation=True, 
        padding="max_length", 
        return_tensors='pt'
    )

    # 5. 复制 input_ids 来生成标签
    labels = encodings["input_ids"].clone()

    # 6. 计算用户输入部分的长度（即需要忽略的部分）
    user_input_len = len(tokenizer(combined_input, truncation=True, max_length=max_length)["input_ids"])

    # 7. 将用户输入部分的标签设置为 -100，忽略这部分的损失
    labels[:, :user_input_len] = -100

    # 8. 返回 input_ids、attention_mask 和 labels，用于训练模型
    return {
        'input_ids': encodings['input_ids'].squeeze(0),  # 输入序列
        'attention_mask': encodings['attention_mask'].squeeze(0),  # 注意力掩码
        'labels': labels.squeeze(0)  # 标签，忽略用户输入部分
    }


In [10]:
# 假设 ds 是你的数据集，并且 tokenizer 已经定义
tokenized_ds = ds.map(
    process_func,
    remove_columns=['instruction', 'input', 'output'],  # 删除不需要的原始列
    fn_kwargs={'tokenizer': tokenizer},  # 将 tokenizer 传递给 process_func
)

tokenized_ds

Map:   0%|          | 0/40924 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10231
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 40924
    })
})

## 3、创建模型

In [11]:
model = AutoModelForCausalLM.from_pretrained(model_path)

### 3.1 查看参数总量

In [13]:
# 计算模型的总参数数量
total_params = sum(param.numel() for param in model.parameters())
print(f"Model size: {total_params / 1e6:.2f}M parameters")  # 将参数数量转换为百万单位显示

Model size: 494.03M parameters


In [16]:
# 打印模型的所有参数名称
for name, param in model.named_parameters():
    print(name)


model.embed_tokens.weight
model.layers.0.self_attn.q_proj.weight
model.layers.0.self_attn.q_proj.bias
model.layers.0.self_attn.k_proj.weight
model.layers.0.self_attn.k_proj.bias
model.layers.0.self_attn.v_proj.weight
model.layers.0.self_attn.v_proj.bias
model.layers.0.self_attn.o_proj.weight
model.layers.0.mlp.gate_proj.weight
model.layers.0.mlp.up_proj.weight
model.layers.0.mlp.down_proj.weight
model.layers.0.input_layernorm.weight
model.layers.0.post_attention_layernorm.weight
model.layers.1.self_attn.q_proj.weight
model.layers.1.self_attn.q_proj.bias
model.layers.1.self_attn.k_proj.weight
model.layers.1.self_attn.k_proj.bias
model.layers.1.self_attn.v_proj.weight
model.layers.1.self_attn.v_proj.bias
model.layers.1.self_attn.o_proj.weight
model.layers.1.mlp.gate_proj.weight
model.layers.1.mlp.up_proj.weight
model.layers.1.mlp.down_proj.weight
model.layers.1.input_layernorm.weight
model.layers.1.post_attention_layernorm.weight
model.layers.2.self_attn.q_proj.weight
model.layers.2.self

### 3.2 选择训练参数

In [22]:
# Freeze方法：冻结大部分参数，只解冻需要微调的部分（例如模型的最后一层和norm层）
num_trainable_params = 0  # 用于统计参与训练的参数数量

# 解冻模型的最后一层和norm层的参数
for name, param in model.named_parameters():
    if "layers.23" in name or "norm" in name:  # 解冻最后一层和norm层
        param.requires_grad = True  # 解冻这些层的参数
        num_trainable_params += param.numel()  # 统计可训练的参数数量
    else:
        param.requires_grad = False  # 冻结其他所有层的参数

# 输出参与训练的参数数量和总参数数量
print(f"Number of trainable parameters (last layer and norm): {num_trainable_params/ 1e6:.2f}M parameters")
print(f"Trainable parameter ratio: {num_trainable_params / total_params:.6f}")


Number of trainable parameters (last layer and norm): 14.95M parameters
Trainable parameter ratio: 0.030270


In [18]:
# 计算模型的内存占用情况
model_size_gb = total_params * 4 / (1024 ** 3)  # 计算模型权重在显存中的占用，按每个参数4字节计算
gradient_size_gb = model_size_gb  # 梯度占用
optimizer_size_gb = model_size_gb * 2  # Adam优化器的状态需要2倍的参数空间

# 输出内存使用情况
print(f"Model size: {model_size_gb:.2f} GB")
print(f"Gradient size: {gradient_size_gb:.2f} GB")
print(f"Optimizer size: {optimizer_size_gb:.2f} GB")
print(f"Total memory usage: {model_size_gb + gradient_size_gb + optimizer_size_gb:.2f} GB")

Model size: 1.84 GB
Gradient size: 1.84 GB
Optimizer size: 3.68 GB
Total memory usage: 7.36 GB


## 5、创建训练参数

In [19]:
args = TrainingArguments(
    output_dir="./models_for_chatbot",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,  # 每4步累计一次
    per_device_eval_batch_size=32,
    logging_steps=20,
    num_train_epochs=1,
    report_to=['tensorboard'],
)

## 6、创建训练器并训练

In [20]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"], 
    eval_dataset=tokenized_ds["test"],     
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
)

In [21]:
trainer.train()

  0%|          | 0/319 [00:00<?, ?it/s]

{'loss': 2.4521, 'grad_norm': 0.7350702881813049, 'learning_rate': 4.686520376175549e-05, 'epoch': 0.06}
{'loss': 0.4851, 'grad_norm': 0.5101087689399719, 'learning_rate': 4.373040752351097e-05, 'epoch': 0.13}
{'loss': 0.5391, 'grad_norm': 0.48463982343673706, 'learning_rate': 4.059561128526646e-05, 'epoch': 0.19}
{'loss': 0.4967, 'grad_norm': 0.5809065699577332, 'learning_rate': 3.746081504702195e-05, 'epoch': 0.25}
{'loss': 0.5146, 'grad_norm': 0.45938950777053833, 'learning_rate': 3.4326018808777435e-05, 'epoch': 0.31}
{'loss': 0.5228, 'grad_norm': 0.5599532723426819, 'learning_rate': 3.119122257053292e-05, 'epoch': 0.38}
{'loss': 0.4858, 'grad_norm': 0.4776483476161957, 'learning_rate': 2.8056426332288405e-05, 'epoch': 0.44}
{'loss': 0.5083, 'grad_norm': 0.6726183891296387, 'learning_rate': 2.4921630094043887e-05, 'epoch': 0.5}
{'loss': 0.5119, 'grad_norm': 0.676257848739624, 'learning_rate': 2.1786833855799376e-05, 'epoch': 0.56}
{'loss': 0.4861, 'grad_norm': 0.4430706202983856, '

TrainOutput(global_step=319, training_loss=0.6290515343597316, metrics={'train_runtime': 740.5524, 'train_samples_per_second': 13.815, 'train_steps_per_second': 0.431, 'total_flos': 5611659152326656.0, 'train_loss': 0.6290515343597316, 'epoch': 0.9976544175136826})

## 7、预测

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

In [ ]:
#ipt = ": {}\n{}".format("简述文章的主要观点：「人工智能如何帮助应对气候变化」", "").strip() + "\n\nAssistant: "

In [ ]:
ipt = "用户: {}\n{}".format("解释气候变化对环境的两个影响", "").strip() + "\n\n机器人: "

In [ ]:
re = pipe(ipt, max_length=256, do_sample=True, )[0]["generated_text"]

Both `max_new_tokens` (=2048) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print(re)

用户: 解释气候变化对环境的两个影响

机器人: 1. 温度升高影响生物多样性。随着温度升高，海平面上升，海洋动物的栖息地正在减少。
2. 可能有海平面上升，导致沿海地区被洪水淹没，或极端天气，导致农作物或树木受到破坏的损失。


In [ ]:
print(ds['train'][5]["output"])

气候变化对环境有几个影响。其中一个影响是全球平均气温上升，导致冰川融化、极端天气事件和海平面上升。另一个影响是由极端天气条件和升温引起的自然栖息地破坏导致的生物多样性减少。
